In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rng = np.random.default_rng(5)
n = 1000
df = pd.DataFrame({
    "AreaSqFt": rng.integers(500, 4000, n),
    "Bedrooms": rng.integers(1, 6, n),
    "Bathrooms": rng.integers(1, 4, n),
    "AgeYears": rng.integers(0, 40, n),
    "DistanceToCityKm": rng.uniform(1, 30, n),
    "Garage": rng.integers(0, 2, n),
})
df["Price"] = (20000 + df["AreaSqFt"] * 120 + df["Bedrooms"] * 8000
               + df["Bathrooms"] * 6000 - df["AgeYears"] * 1200
               - df["DistanceToCityKm"] * 2500 + df["Garage"] * 10000
               + rng.normal(0, 15000, n))
df.to_csv("house_price.csv", index=False)

df = pd.read_csv("house_price.csv")
print(df.describe().round(2), "\n")

X = df.drop("Price", axis=1)
y = df["Price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LinearRegression().fit(X_train_s, y_train)
rf = RandomForestRegressor(n_estimators=200, random_state=42).fit(X_train, y_train)

for name, pred in [("Linear Regression", lr.predict(X_test_s)),
                   ("Random Forest", rf.predict(X_test))]:
    print(f"{name}: R2 = {r2_score(y_test, pred):.4f}, "
          f"MAE = {mean_absolute_error(y_test, pred):.0f}, "
          f"RMSE = {np.sqrt(mean_squared_error(y_test, pred)):.0f}")

new_house = pd.DataFrame([{"AreaSqFt": 1800, "Bedrooms": 3, "Bathrooms": 2,
                           "AgeYears": 5, "DistanceToCityKm": 8, "Garage": 1}])
print("\nPredicted price (Linear Regression): Rs.", round(lr.predict(scaler.transform(new_house))[0]))
print("Predicted price (Random Forest)    : Rs.", round(rf.predict(new_house)[0]))

plt.scatter(y_test, lr.predict(X_test_s), s=10)
plt.plot([y.min(), y.max()], [y.min(), y.max()], "r--")
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.title("House Price Prediction (Linear Regression)")
plt.show()